In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Combinando Búsqueda Semántica y por Palabras Clave: Un Tutorial de Búsqueda Híbrida con Vertex AI Vector Search

Vector Search soporta búsqueda híbrida, un patrón de arquitectura popular en la recuperación de información (IR) que combina la búsqueda semántica y la búsqueda por palabras clave (también llamada búsqueda basada en tokens). Con la búsqueda híbrida, los desarrolladores pueden aprovechar lo mejor de ambos enfoques, proporcionando efectivamente una mayor calidad de búsqueda.

Este tutorial explica los conceptos de búsqueda híbrida, búsqueda semántica y búsqueda basada en tokens, e incluye ejemplos de cómo configurar la búsqueda basada en tokens y la búsqueda híbrida.

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/embeddings/hybrid-search.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Abrir en Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fembeddings%2Fhybrid-search.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Abrir en Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/embeddings/hybrid-search.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Abrir en Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/hybrid-search.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo"><br> Ver en GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

# ¿Por qué es importante la búsqueda híbrida?

Como se describe en la [Descripción general de Vector Search](https://cloud.google.com/vertex-ai/docs/vector-search/overview), la búsqueda semántica con Vector Search puede encontrar elementos con similitud semántica mediante consultas.

Los modelos de embedding como [Vertex AI embeddings](https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings) construyen un espacio vectorial como un mapa de los significados del contenido. Cada embedding textual o multimodal es una ubicación en el mapa que representa el significado de algún contenido. Vector Search permite encontrar rápidamente otros embeddings en su vecindad. Esta búsqueda por el significado del contenido se denomina búsqueda semántica.

## ¿Por qué combinar la búsqueda semántica con la búsqueda basada en palabras clave?

La búsqueda semántica no cubre todos los requisitos posibles para las aplicaciones de recuperación de información, como la [Generación Aumentada por Recuperación (RAG)](https://cloud.google.com/use-cases/retrieval-augmented-generation). La búsqueda semántica solo puede encontrar datos que el modelo de embedding pueda comprender. Por ejemplo, consultas o conjuntos de datos con números de producto arbitrarios o SKUs, nombres de productos nuevos añadidos recientemente y nombres en clave internos de la empresa no funcionan bien con la búsqueda semántica porque no estaban incluidos en el conjunto de entrenamiento del modelo. Esto se denomina datos "fuera de dominio".

En tales casos, es necesario combinar la búsqueda semántica con la búsqueda basada en palabras clave (también llamada basada en tokens) para formar una búsqueda híbrida. Con la búsqueda híbrida, obtienes lo mejor de ambos mundos para lograr una mayor calidad de búsqueda.

Uno de los sistemas de búsqueda híbrida más populares es la Búsqueda de Google (Google Search). El servicio incorporó la [búsqueda semántica en 2015 con el modelo RankBrain](https://blog.google/products/search/how-ai-powers-great-search-results/), además de su algoritmo de búsqueda por palabras clave basado en tokens.

# Sección 1. Crear embeddings dispersos (sparse embeddings)

En esta primera sección, generarás embeddings dispersos y construirás un endpoint de Vector Search.

## Tarea 2. Instalar paquetes y configurar el notebook.

El notebook utiliza el SDK de Vertex AI y el SDK de Cloud Storage.

In [ ]:
%pip install --upgrade --quiet --user google-cloud-aiplatform google-cloud-storage scikit-learn pandas

In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

In [ ]:
PROJECT_ID = "TU_PROYECTO_ID"
LOCATION = "us-east1"
print(f"Project ID: {PROJECT_ID}")
print(f"Region: {LOCATION}")

## Tarea 3. Preparar un conjunto de datos de ejemplo

Prepararás un archivo de datos para construir un índice para embeddings dispersos.

In [ ]:
import pandas as pd

CSV_URL = "https://storage.googleapis.com/spls/gsp1297/google_merch_shop_items.csv"
df = pd.read_csv(CSV_URL)
df.head()

## Tarea 4. Crear embeddings dispersos

Crearás embeddings dispersos con un vectorizador TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df.title.tolist()
vectorizer = TfidfVectorizer()
vectorizer.fit_transform(corpus)

In [ ]:
def get_sparse_embedding(text):
    tfidf_vector = vectorizer.transform([text])
    values = []
    dims = []
    for i, tfidf_value in enumerate(tfidf_vector.data):
        values.append(float(tfidf_value))
        dims.append(int(tfidf_vector.indices[i]))
    return {"values": values, "dimensions": dims}

# Sección 2. Usar búsqueda híbrida

En esta sección combinarás la búsqueda basada en tokens con la búsqueda semántica.

## Tarea 6. Crear el índice híbrido y desplegarlo

Para construir un índice híbrido, cada elemento debe tener tanto `embedding` (denso) como `sparse_embedding`.

In [ ]:
from vertexai.language_models import TextEmbeddingModel

model = TextEmbeddingModel.from_pretrained("text-embedding-004")

def get_dense_embedding(text):
    return model.get_embeddings([text])[0].values

get_dense_embedding("Chrome Dino Pin")

## Tarea 7. Ejecutar una consulta híbrida

Después de desplegar el índice, puedes ejecutar una consulta híbrida.

In [ ]:
from google.cloud.aiplatform.matching_engine.matching_engine_index_endpoint import HybridQuery

query_text = "Kids"
query_dense_emb = get_dense_embedding(query_text)
query_sparse_emb = get_sparse_embedding(query_text)

query = HybridQuery(
    dense_embedding=query_dense_emb,
    sparse_embedding_dimensions=query_sparse_emb["dimensions"],
    sparse_embedding_values=query_sparse_emb["values"],
    rrf_ranking_alpha=0.5,
)

En una búsqueda híbrida se utiliza **Reciprocal Rank Fusion (RRF)** para combinar los resultados de la búsqueda semántica y la búsqueda por tokens.

# Conclusión

En este tutorial hemos aprendido el concepto de búsqueda híbrida y cómo combinar embeddings densos y dispersos en un solo índice de Vector Search.

# Conceptos Adicionales

## ¿Cómo funcionan TF-IDF y `TfidfVectorizer`?
TF-IDF intenta dar más peso a las palabras distintivas del conjunto de datos (como "Shirts" o "Dino") en comparación con palabras triviales (como "the", "a", "of").

## ¿Qué es Reciprocal Rank Fusion (RRF)?
RRF es un algoritmo para combinar múltiples listas ordenadas en una única clasificación unificada. Es ideal para fusionar resultados de búsquedas semánticas y por palabras clave.